# MDPs in Python: the Golden Goose, the Wavering Customer, Double or Nothing

This morning Old Bessie showed what every MDP has: states, actions and transition dynamics. Here you write three
small MDPs in Python yourself. In each of them the transition dynamics are one function: given a state and an
action, it gives back the next state and the money, the reward. Task A is a goose that lays golden eggs, task B a
customer who may leave, task C a quiz of double or nothing.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one
only if you are still stuck. It takes about 30 to 40 minutes.

Start with the two folded cells below: the kit and the checks.

In [ ]:
#@title The kit (run me first) { display-mode: "form" }
# The kit of the MDPs notebook. Run this cell first; you do not need to read it.
import inspect as _kit_inspect
import random as _kit_random

_KIT = {"luck": _kit_random.Random(14), "spins": 0, "forced": None}
_KIT_SAME_LUCK = 3        # every average_money plays the same luck
_KIT_TRIES = 10000               # how many times average_money plays a strategy


def spin():
    """The ball of a roulette: a number between 0 and 1, every number as likely as any other."""
    _KIT["spins"] = _KIT["spins"] + 1
    if _KIT["forced"] is not None:
        return _KIT["forced"]
    return _KIT["luck"].random()


# The three MDPs, known by the names of their functions
_KIT_MDPS = {
    "goose_transition": {
        "states": ["GOOSE", "SOLD"], "actions": ["KEEP", "SELL"], "task": "A", "turn": "week", "end": 1,
        "check": "check_goose(goose_transition)", "ended": "The goose is SOLD: nothing more happens.",
        "example": "[SELL, SELL]"},
    "customer_transition": {
        "states": ["LOYAL", "WAVERING", "GONE"], "actions": ["FULL_PRICE", "DISCOUNT"], "task": "B", "turn": "week",
        "end": 2, "check": "check_customer(customer_transition)",
        "ended": "The customer is GONE: nothing more happens.", "example": "[DISCOUNT, DISCOUNT, DISCOUNT]"},
    "quiz_transition": {
        "states": ["Q1", "Q2", "Q3", "Q4", "OVER"], "actions": ["ANSWER", "STOP"], "task": "C",
        "turn": "question", "end": 4, "check": "check_quiz(quiz_transition)", "ended": "The game is OVER.",
        "example": "[ANSWER, STOP, STOP, STOP, STOP]"},
}


def _kit_mdp(transition):
    return _KIT_MDPS.get(getattr(transition, "__name__", ""))


def _kit_is_int(x):
    return isinstance(x, int) and not isinstance(x, bool)


def _kit_state(mdp, s):
    if _kit_is_int(s) and 0 <= s < len(mdp["states"]):
        return mdp["states"][s]
    return repr(s)


def _kit_action(mdp, a):
    if _kit_is_int(a) and 0 <= a < len(mdp["actions"]):
        return mdp["actions"][a]
    return repr(a)


def _kit_money(x):
    """$10 for a whole number, $103.95 otherwise."""
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        return repr(x)
    if x == int(x):
        return "$" + format(int(x), ",") if x >= 0 else "-$" + format(-int(x), ",")
    return "$" + format(x, ",.2f") if x >= 0 else "-$" + format(-x, ",.2f")


def _kit_gaps(fn):
    """How many lines of the function are still a gap: a line that starts with ..."""
    try:
        src = _kit_inspect.getsource(fn)
    except (OSError, TypeError):
        return 0
    n = 0
    for line in src.splitlines():
        if line.strip().startswith("..."):
            n = n + 1
    return n


def _kit_which(transition):
    """The MDP of a function, or None after a calm line."""
    mdp = _kit_mdp(transition)
    if mdp is None:
        print("⏳ give one of the three functions of the tasks: goose_transition, customer_transition or "
              "quiz_transition.")
    return mdp


def _kit_call(transition, state, action):
    """Calls the participant's function calmly: (next_state, money), or None after one printed line."""
    mdp = _kit_which(transition)
    if mdp is None:
        return None
    name = transition.__name__
    if _kit_gaps(transition) > 0:
        print("⏳ %s is not finished yet: fill in the 🎯 gaps of task %s first." % (name, mdp["task"]))
        return None
    call = "%s(%s, %s)" % (name, _kit_state(mdp, state), _kit_action(mdp, action))
    try:
        out = transition(state, action)
    except Exception as e:
        print("❌ %s stops with an error: %s: %s. %s says more." % (call, type(e).__name__, e, mdp["check"]))
        return None
    if not (isinstance(out, (tuple, list)) and len(out) == 2):
        print("❌ %s gives back %r: it should give back the next state and the money. %s says more."
              % (call, out, mdp["check"]))
        return None
    return out[0], out[1]


def one_week(transition, state, action):
    """Calls transition(state, action) once and prints the next state and the money, with their names."""
    out = _kit_call(transition, state, action)
    if out is None:
        return
    mdp = _kit_mdp(transition)
    print("%s(%s, %s) gives back (%r, %r)" % (transition.__name__, _kit_state(mdp, state),
                                             _kit_action(mdp, action), out[0], out[1]))
    print("   the next state: %s      the money: %s" % (_kit_state(mdp, out[0]), _kit_money(out[1])))


def many_times(transition, state, action, times=1000):
    """Calls transition(state, action) many times, a new ball each time, and counts what happens."""
    out = _kit_call(transition, state, action)          # the first call, calmly
    if out is None:
        return
    seen = []
    counts = []
    for i in range(times):
        if i > 0:
            try:
                out = transition(state, action)
                out = (out[0], out[1])
            except Exception:
                out = _kit_call(transition, state, action)
                if out is None:
                    return
        if out in seen:
            counts[seen.index(out)] = counts[seen.index(out)] + 1
        else:
            seen.append(out)
            counts.append(1)
    mdp = _kit_mdp(transition)
    print("%s times %s(%s, %s):" % (format(times, ","), transition.__name__, _kit_state(mdp, state),
                                    _kit_action(mdp, action)))
    order = sorted(range(len(seen)), key=lambda i: (str(seen[i][0]), -counts[i]))
    for i in order:
        s, m = seen[i]
        print("   next state %-9s money %-6s %6s times" % (_kit_state(mdp, s), _kit_money(m), format(counts[i], ",")))


def play_turns(transition, start, actions):
    """Plays the actions of the list one after the other, each turn from the state the turn before left."""
    mdp = _kit_which(transition)
    if mdp is None:
        return
    if not isinstance(actions, (list, tuple)):
        print("❌ actions should be a list, such as [%s, %s]." % (mdp["actions"][0], mdp["actions"][1]))
        return
    state = start
    total = 0
    turn = 0
    for action in actions:
        if state == mdp["end"]:
            break
        turn = turn + 1
        out = _kit_call(transition, state, action)
        if out is None:
            return
        print("%s %2d   %-9s %-11s ->  next state %-9s money %s" % (
            mdp["turn"], turn, _kit_state(mdp, state), _kit_action(mdp, action), _kit_state(mdp, out[0]),
            _kit_money(out[1])))
        state = out[0]
        total = total + out[1]
    if state == mdp["end"]:
        print(mdp["ended"])
    print("money in all: %s" % _kit_money(total))


def _kit_strategy_ok(mdp, strategy):
    if not isinstance(strategy, (list, tuple)) or len(strategy) != len(mdp["states"]):
        return False
    for a in strategy:
        if not (_kit_is_int(a) and 0 <= a < len(mdp["actions"])):
            return False
    return True


def average_money(transition, start, strategy, turns=13):
    """Plays a strategy many times, every time from start, and prints the average money of a try. Every strategy
    meets the same luck."""
    mdp = _kit_which(transition)
    if mdp is None:
        return
    try:
        ready = _kit_check(transition.__name__, transition, quiet=True)
    except NameError:
        print("⏳ run the folded cell of the checks first.")
        return
    if not ready:
        print("⏳ finish task %s first: %s says what is missing." % (mdp["task"], mdp["check"]))
        return
    if not _kit_strategy_ok(mdp, strategy):
        print("❌ a strategy gives one action for each of the %d states, in the order %s: for example %s."
              % (len(mdp["states"]), ", ".join(mdp["states"]), mdp["example"]))
        return
    saved = _KIT["luck"]
    _KIT["luck"] = _kit_random.Random(_KIT_SAME_LUCK)
    total = 0
    try:
        for t in range(_KIT_TRIES):
            state = start
            for turn in range(turns):
                if state == mdp["end"]:
                    break
                state, money = transition(state, strategy[state])
                total = total + money
    finally:
        _KIT["luck"] = saved
    label = "[" + ", ".join(_kit_action(mdp, a) for a in strategy) + "]"
    if mdp["turn"] == "question":
        what = "%s tries of the quiz" % format(_KIT_TRIES, ",")
    else:
        what = "%s tries of %d weeks" % (format(_KIT_TRIES, ","), turns)
    print("%-41s $%s on average over %s" % (label, format(total / _KIT_TRIES, ",.2f"), what))


print("✅ The kit is ready: spin, one_week, many_times, play_turns, average_money.")

In [ ]:
#@title The checks (run me too) { display-mode: "form" }
# The checks of the MDPs notebook. Run this cell too; you do not need to read it.

# Each test: state, action, ball, the next state and the money the table gives, whether the ball decides, the row
_KIT_TESTS = {
    'goose_transition': [
        (0, 0, 0.02, 0, 10, True, 'a ball below 0.8 brings a golden egg, $10, and the goose stays'),
        (0, 0, 0.79, 0, 10, True, 'a ball below 0.8 brings a golden egg, $10, and the goose stays'),
        (0, 0, 0.81, 0, 0, True, 'a ball of 0.8 or more brings no egg, $0, and the goose stays'),
        (0, 0, 0.98, 0, 0, True, 'a ball of 0.8 or more brings no egg, $0, and the goose stays'),
        (0, 1, 0.3, 1, 50, False, 'the butcher pays $50, and the goose is SOLD'),
        (0, 1, 0.95, 1, 50, False, 'the butcher pays $50, and the goose is SOLD'),
        (1, 0, 0.3, 1, 0, False, 'a SOLD goose brings nothing more and stays SOLD'),
        (1, 0, 0.95, 1, 0, False, 'a SOLD goose brings nothing more and stays SOLD'),
        (1, 1, 0.3, 1, 0, False, 'a SOLD goose brings nothing more and stays SOLD'),
        (1, 1, 0.95, 1, 0, False, 'a SOLD goose brings nothing more and stays SOLD'),
    ],
    'customer_transition': [
        (0, 0, 0.02, 1, 10, True, 'a LOYAL customer at FULL_PRICE pays $10 and starts to waver when the ball is below 0.3'),
        (0, 0, 0.29, 1, 10, True, 'a LOYAL customer at FULL_PRICE pays $10 and starts to waver when the ball is below 0.3'),
        (0, 0, 0.31, 0, 10, True, 'a LOYAL customer at FULL_PRICE pays $10 and stays LOYAL when the ball is 0.3 or more'),
        (0, 0, 0.98, 0, 10, True, 'a LOYAL customer at FULL_PRICE pays $10 and stays LOYAL when the ball is 0.3 or more'),
        (0, 1, 0.3, 0, 7, False, 'a DISCOUNT brings $7 and keeps the customer LOYAL'),
        (0, 1, 0.95, 0, 7, False, 'a DISCOUNT brings $7 and keeps the customer LOYAL'),
        (1, 0, 0.02, 2, 10, True, 'a WAVERING customer at FULL_PRICE pays $10 and leaves, GONE, when the ball is below 0.5'),
        (1, 0, 0.49, 2, 10, True, 'a WAVERING customer at FULL_PRICE pays $10 and leaves, GONE, when the ball is below 0.5'),
        (1, 0, 0.51, 1, 10, True, 'a WAVERING customer at FULL_PRICE pays $10 and keeps WAVERING when the ball is 0.5 or more'),
        (1, 0, 0.98, 1, 10, True, 'a WAVERING customer at FULL_PRICE pays $10 and keeps WAVERING when the ball is 0.5 or more'),
        (1, 1, 0.3, 0, 7, False, 'a DISCOUNT brings $7 and wins the customer back: LOYAL'),
        (1, 1, 0.95, 0, 7, False, 'a DISCOUNT brings $7 and wins the customer back: LOYAL'),
        (2, 0, 0.3, 2, 0, False, 'a customer who is GONE pays nothing and stays GONE'),
        (2, 0, 0.95, 2, 0, False, 'a customer who is GONE pays nothing and stays GONE'),
        (2, 1, 0.3, 2, 0, False, 'a customer who is GONE pays nothing and stays GONE'),
        (2, 1, 0.95, 2, 0, False, 'a customer who is GONE pays nothing and stays GONE'),
    ],
    'quiz_transition': [
        (0, 0, 0.02, 1, 0, True, 'a ball below 0.9 at Q1 is a right answer: on to Q2, no money yet'),
        (0, 0, 0.89, 1, 0, True, 'a ball below 0.9 at Q1 is a right answer: on to Q2, no money yet'),
        (0, 0, 0.91, 4, 0, True, 'a ball of 0.9 or more at Q1 is a wrong answer: nothing, and the game is OVER'),
        (0, 0, 0.98, 4, 0, True, 'a ball of 0.9 or more at Q1 is a wrong answer: nothing, and the game is OVER'),
        (0, 1, 0.3, 4, 0, False, 'STOP takes the bank of Q1 home, $0, and the game is OVER'),
        (0, 1, 0.95, 4, 0, False, 'STOP takes the bank of Q1 home, $0, and the game is OVER'),
        (1, 0, 0.02, 2, 0, True, 'a ball below 0.8 at Q2 is a right answer: on to Q3, no money yet'),
        (1, 0, 0.79, 2, 0, True, 'a ball below 0.8 at Q2 is a right answer: on to Q3, no money yet'),
        (1, 0, 0.81, 4, 0, True, 'a ball of 0.8 or more at Q2 is a wrong answer: nothing, and the game is OVER'),
        (1, 0, 0.98, 4, 0, True, 'a ball of 0.8 or more at Q2 is a wrong answer: nothing, and the game is OVER'),
        (1, 1, 0.3, 4, 100, False, 'STOP takes the bank of Q2 home, $100, and the game is OVER'),
        (1, 1, 0.95, 4, 100, False, 'STOP takes the bank of Q2 home, $100, and the game is OVER'),
        (2, 0, 0.02, 3, 0, True, 'a ball below 0.6 at Q3 is a right answer: on to Q4, no money yet'),
        (2, 0, 0.59, 3, 0, True, 'a ball below 0.6 at Q3 is a right answer: on to Q4, no money yet'),
        (2, 0, 0.61, 4, 0, True, 'a ball of 0.6 or more at Q3 is a wrong answer: nothing, and the game is OVER'),
        (2, 0, 0.98, 4, 0, True, 'a ball of 0.6 or more at Q3 is a wrong answer: nothing, and the game is OVER'),
        (2, 1, 0.3, 4, 200, False, 'STOP takes the bank of Q3 home, $200, and the game is OVER'),
        (2, 1, 0.95, 4, 200, False, 'STOP takes the bank of Q3 home, $200, and the game is OVER'),
        (3, 0, 0.02, 4, 800, True, 'a ball below 0.4 at Q4 is a right answer: $800, and the game is OVER'),
        (3, 0, 0.39, 4, 800, True, 'a ball below 0.4 at Q4 is a right answer: $800, and the game is OVER'),
        (3, 0, 0.41, 4, 0, True, 'a ball of 0.4 or more at Q4 is a wrong answer: nothing, and the game is OVER'),
        (3, 0, 0.98, 4, 0, True, 'a ball of 0.4 or more at Q4 is a wrong answer: nothing, and the game is OVER'),
        (3, 1, 0.3, 4, 400, False, 'STOP takes the bank of Q4 home, $400, and the game is OVER'),
        (3, 1, 0.95, 4, 400, False, 'STOP takes the bank of Q4 home, $400, and the game is OVER'),
        (4, 0, 0.3, 4, 0, False, 'when the game is OVER, nothing more happens'),
        (4, 0, 0.95, 4, 0, False, 'when the game is OVER, nothing more happens'),
        (4, 1, 0.3, 4, 0, False, 'when the game is OVER, nothing more happens'),
        (4, 1, 0.95, 4, 0, False, 'when the game is OVER, nothing more happens'),
    ],
}


def _kit_check(key, transition, quiet=False):
    """Runs the tests of one MDP. True when all pass; prints one line unless quiet."""
    mdp = _KIT_MDPS[key]

    def say(line):
        if not quiet:
            print(line)

    if not callable(transition):
        say("⏳ %s is not a function yet: run the cell of task %s first." % (key, mdp["task"]))
        return False
    gaps = _kit_gaps(transition)
    if gaps > 0:
        say("⏳ %s is not finished yet: replace every ... of task %s with your code (%d left)."
            % (key, mdp["task"], gaps))
        return False
    tests = _KIT_TESTS[key]
    for state, action, ball, want_state, want_money, chance, why in tests:
        call = "%s(%s, %s)" % (key, _kit_state(mdp, state), _kit_action(mdp, action))
        if chance:
            call = call + " with the ball %s" % ball
        _KIT["forced"] = ball
        _KIT["spins"] = 0
        try:
            out = transition(state, action)
        except Exception as e:
            say("❌ %s stops with an error: %s: %s" % (call, type(e).__name__, e))
            return False
        finally:
            _KIT["forced"] = None
        spins = _KIT["spins"]
        want = "(%s, %s)" % (_kit_state(mdp, want_state), _kit_money(want_money))
        if out is None:
            say("❌ %s gives back nothing (None). Every way through the function ends in a return of two values, "
                "the next state and the money. The table says %s: %s." % (call, want, why))
            return False
        if not (isinstance(out, (tuple, list)) and len(out) == 2):
            say("❌ %s gives back %r. It should give back two values, the next state and the money: the table "
                "says %s, so return %s, %s." % (call, out, want, mdp["states"][want_state], want_money))
            return False
        if isinstance(out[0], str):
            say("❌ %s gives back %r. A state is a number: write %s without quotes."
                % (call, tuple(out), out[0]))
            return False
        if spins != 1:
            say("❌ %s spins %d balls in one %s. A %s has one ball: keep the first line, ball = spin(), and use "
                "ball below it." % (key, spins, mdp["turn"], mdp["turn"]))
            return False
        if out[0] != want_state or out[1] != want_money:
            say("❌ %s gives back (%s, %s). The table says %s: %s."
                % (call, _kit_state(mdp, out[0]), _kit_money(out[1]), want, why))
            return False
    say("✅ %s follows the table in all %d tests. Task %s is done." % (key, len(tests), mdp["task"]))
    return True


def check_goose(goose_transition):
    _kit_check("goose_transition", goose_transition)


def check_customer(customer_transition):
    _kit_check("customer_transition", customer_transition)


def check_quiz(quiz_transition):
    _kit_check("quiz_transition", quiz_transition)


print("✅ The checks are ready: check_goose, check_customer, check_quiz.")

## Luck in Python

Transition dynamics are usually random: in Old Bessie, the same state and the same action can lead to different
next states. In Python the luck comes from `spin()`, the kit's roulette ball: a number between 0 and 1, every number as
likely as any other. Something that happens with chance 0.8 happens when the ball lands below 0.8.

💭 **Think first.** Of 1,000 balls, how many land below 0.8?

In [ ]:
ball = spin()
print("one ball:", ball)

below = 0
for i in range(1000):
    if spin() < 0.8:
        below = below + 1
print(below, "of 1,000 balls landed below 0.8")

**Reading the result.** About 800, a few more or fewer each time you run the cell. So `if ball < 0.8:` is how
Python says "with chance 0.8". Every transition function of this notebook starts with the line `ball = spin()`:
one ball for the turn, which the rules then read.

## The Golden Goose

A farmer owns a goose that lays a golden egg, worth \$10, in 80% of the weeks. A butcher offers \$50 for the
goose. Every week the farmer chooses: KEEP the goose, or SELL it. A sold goose is gone for good.

| State | Action | Chance | Next state | Money |
|---|---|---|---|---|
| GOOSE | KEEP | 0.8 | GOOSE | \$10 |
| GOOSE | KEEP | 0.2 | GOOSE | \$0 |
| GOOSE | SELL | 1 | SOLD | \$50 |
| SOLD | KEEP or SELL | 1 | SOLD | \$0 |

This table is the whole MDP: two states, two actions, and the transition dynamics, one row per outcome. In Python
the states and the actions are numbers, with names that read like the story.

In [ ]:
GOOSE, SOLD = 0, 1          # the states
KEEP, SELL = 0, 1           # the actions

## 🎯 Task A. The goose's transition dynamics

Write `goose_transition(state, action)`: it gives back the state next week and the week's money, as the table
says. A line such as `return GOOSE, 10` gives back both, first the next state, then the money. The first line
spins the week's ball, and the rows of SOLD are written for you.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Each 🎯 line becomes one return, and each return is one row of the table: the next state, then the money.
# The row of SELL has chance 1, the next state SOLD and the money $50.

In [ ]:
#@title 💡 Hint 2 for task A: the three lines (double-click to open) { display-mode: "form" }
#         return SOLD, 50
#             return GOOSE, 10
#         return GOOSE, 0

In [ ]:
def goose_transition(state, action):
    """One week of the goose: gives back the state next week and the week's money."""
    ball = spin()                          # the week's ball: a number between 0 and 1
    if state == SOLD:
        return SOLD, 0                     # a sold goose brings nothing more, whatever the action
    if action == SELL:
        ...                                # 🎯 YOUR TURN: the butcher pays $50, and the goose is SOLD
    if action == KEEP:
        if ball < 0.8:
            ...                            # 🎯 YOUR TURN: a golden egg, $10, and the goose stays
        ...                                # 🎯 YOUR TURN: no egg, $0, and the goose stays

In [ ]:
check_goose(goose_transition)

## One week

Your function is the goose's transition dynamics. `one_week` calls it once and prints what it gives back, with
the names. Run the cell a few times: each run spins a new ball.

In [ ]:
one_week(goose_transition, GOOSE, KEEP)

**Reading the result.** The first line is what your function gives back: two numbers, the next state and the
money. The second line names the state: 0 is GOOSE. About 8 runs in 10 bring an egg.

## Weeks by hand

You are the farmer. Write your actions, one per week, in the list `actions`: `play_turns` plays them one after the
other, each week from the state the week before left, and adds up the money.

In [ ]:
actions = [KEEP, KEEP, KEEP, SELL, KEEP]          # change me: one action per week
play_turns(goose_transition, GOOSE, actions)

**Reading the result.** After week 4 the goose is SOLD: the butcher's \$50 came once, and the fifth action changes
nothing. Try other lists: sell at once, or keep the goose for all five weeks.

## The Wavering Customer

A customer buys from you every week. At FULL_PRICE they pay \$10, but a LOYAL customer starts to waver in 30% of
the weeks, and a WAVERING customer leaves for good in 50% of the weeks, after paying. A DISCOUNT brings \$7 and
makes the customer LOYAL next week.

| State | Action | Chance | Next state | Money |
|---|---|---|---|---|
| LOYAL | FULL_PRICE | 0.3 | WAVERING | \$10 |
| LOYAL | FULL_PRICE | 0.7 | LOYAL | \$10 |
| LOYAL | DISCOUNT | 1 | LOYAL | \$7 |
| WAVERING | FULL_PRICE | 0.5 | GONE | \$10 |
| WAVERING | FULL_PRICE | 0.5 | WAVERING | \$10 |
| WAVERING | DISCOUNT | 1 | LOYAL | \$7 |
| GONE | FULL_PRICE or DISCOUNT | 1 | GONE | \$0 |

In [ ]:
LOYAL, WAVERING, GONE = 0, 1, 2       # the states
FULL_PRICE, DISCOUNT = 0, 1           # the actions

## 🎯 Task B. The customer's transition dynamics

Write `customer_transition(state, action)`, one block per state and action. The first block, LOYAL and
FULL_PRICE, is written for you: two rows of the table, with chances 0.3 and 0.7, become an `if` on the ball. The
first of the two rows takes the balls below its chance, the second row the rest. A row with chance 1 is a single
return.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# LOYAL and DISCOUNT is one row with chance 1: a single return, the next state LOYAL and the money 7.
# WAVERING and FULL_PRICE is two rows, as in the first block: an if on the ball with 0.5, a return inside it,
# and a return after it.

In [ ]:
#@title 💡 Hint 2 for task B: the lines (double-click to open) { display-mode: "form" }
#         return LOYAL, 7
#
#         if ball < 0.5:
#             return GONE, 10
#         return WAVERING, 10
#
#         return LOYAL, 7
#
#         return GONE, 0

In [ ]:
def customer_transition(state, action):
    """One week of the customer: gives back the customer's state next week and the week's money."""
    ball = spin()                          # the week's ball
    if state == LOYAL and action == FULL_PRICE:
        if ball < 0.3:
            return WAVERING, 10            # pays $10, then starts to waver: chance 0.3
        return LOYAL, 10                   # pays $10 and stays loyal: chance 0.7
    if state == LOYAL and action == DISCOUNT:
        ...                                # 🎯 YOUR TURN: one row of the table, chance 1
    if state == WAVERING and action == FULL_PRICE:
        ...                                # 🎯 YOUR TURN: two rows, chances 0.5 and 0.5, as in the first block
    if state == WAVERING and action == DISCOUNT:
        ...                                # 🎯 YOUR TURN
    if state == GONE:
        ...                                # 🎯 YOUR TURN: the same for both actions

In [ ]:
check_customer(customer_transition)

## A thousand weeks

💭 **Think first.** A WAVERING customer meets FULL_PRICE in 1,000 separate weeks, each with a new ball. In how many
of them does the customer leave?

In [ ]:
many_times(customer_transition, WAVERING, FULL_PRICE)

**Reading the result.** About 500 GONE and 500 still WAVERING: the counts follow the chances of the table. Try
another state and action, such as `LOYAL, FULL_PRICE`.

## Double or Nothing

A quiz of four questions, Q1 to Q4. Before each question you STOP and take the bank home, or you ANSWER. The bank
is \$0 before Q1, \$100 before Q2, \$200 before Q3 and \$400 before Q4, and a right answer at Q4 wins \$800. A wrong
answer ends the game with nothing. The questions get harder: you answer right with chance 0.9, 0.8, 0.6 and 0.4.

| State | Action | Chance | Next state | Money |
|---|---|---|---|---|
| Q1, Q2, Q3, Q4 | STOP | 1 | OVER | the bank: \$0, \$100, \$200, \$400 |
| Q1, Q2, Q3 | ANSWER | right: 0.9, 0.8, 0.6 | the next question | \$0 |
| Q4 | ANSWER | right: 0.4 | OVER | \$800 |
| Q1, Q2, Q3, Q4 | ANSWER | wrong: 0.1, 0.2, 0.4, 0.6 | OVER | \$0 |
| OVER | ANSWER or STOP | 1 | OVER | \$0 |

The four questions follow the same rules with other numbers. Two lists hold the numbers, and the state, itself a
number, picks from them: `BANK[Q3]` is the bank at Q3.

In [ ]:
Q1, Q2, Q3, Q4, OVER = 0, 1, 2, 3, 4      # the states: the question you face, or the game is over
ANSWER, STOP = 0, 1                        # the actions
BANK = [0, 100, 200, 400]                  # what STOP takes home at Q1, Q2, Q3, Q4
CHANCE = [0.9, 0.8, 0.6, 0.4]              # the chance of a right answer at Q1, Q2, Q3, Q4

print("At Q3 the bank is", BANK[Q3], "and a right answer has chance", CHANCE[Q3])

## 🎯 Task C. The quiz's transition dynamics

Write `quiz_transition(state, action)`. `BANK[state]` is the bank of the question you face, and `state + 1` is the
next question: `Q3 + 1` is `Q4`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# STOP: the next state is OVER, and the money is the bank of this question, BANK[state].
# A right answer at Q4 gives OVER and 800; at another question, the next question, state + 1, and 0.
# A wrong answer gives OVER and 0.

In [ ]:
#@title 💡 Hint 2 for task C: the four lines (double-click to open) { display-mode: "form" }
#         return OVER, BANK[state]
#             return OVER, 800
#         return state + 1, 0
#     return OVER, 0

In [ ]:
def quiz_transition(state, action):
    """One question: gives back the state after it and the money taken home."""
    ball = spin()                          # the question's ball
    if state == OVER:
        return OVER, 0                     # the game is over: nothing more happens
    if action == STOP:
        ...                                # 🎯 YOUR TURN: take the bank of this question home; the game is over
    if ball < CHANCE[state]:               # a right answer
        if state == Q4:
            ...                            # 🎯 YOUR TURN: right at the last question: $800, and the game is over
        ...                                # 🎯 YOUR TURN: on to the next question, no money yet
    ...                                    # 🎯 YOUR TURN: a wrong answer: nothing, and the game is over

In [ ]:
check_quiz(quiz_transition)

## A game by hand

Play the quiz: write your actions in the list. The game ends at your first STOP or your first wrong answer.

In [ ]:
actions = [ANSWER, ANSWER, STOP, STOP]            # change me: one action per question
play_turns(quiz_transition, Q1, actions)

## Which way of playing earns the most?

A **strategy** says which action to take in each state: a list with one action per state, in the order of the
states. `[SELL, SELL]` sells the goose at once; its second action, for SOLD, changes nothing. `average_money`
plays a strategy 10,000 times with your transition dynamics, each time from the start, and prints the average
money. Every strategy meets the same sequence of balls, so every correct function prints the averages quoted
below, to the cent.

💭 **Think first.** SELL brings \$50 at once, more than any week of eggs. Over a quarter of 13 weeks, which earns
more: selling at once, or keeping the goose?

In [ ]:
average_money(goose_transition, GOOSE, [SELL, SELL])   # sell at once
average_money(goose_transition, GOOSE, [KEEP, KEEP])   # keep the goose every week

**Reading the result.** Keeping the goose brings \$104.08 over a quarter, about twice the butcher's \$50. With fewer
weeks to go the answer changes: here are 6 weeks.

In [ ]:
average_money(goose_transition, GOOSE, [SELL, SELL], turns=6)
average_money(goose_transition, GOOSE, [KEEP, KEEP], turns=6)

**Reading the result.** Six weeks of eggs bring \$47.78 on average, less than \$50: with 6 weeks to go, sell.

💭 **Think first.** Full price brings \$10 a week, a discount \$7. Which strategy earns the most over a quarter?

In [ ]:
average_money(customer_transition, LOYAL, [FULL_PRICE, FULL_PRICE, FULL_PRICE])   # always full price
average_money(customer_transition, LOYAL, [DISCOUNT, DISCOUNT, DISCOUNT])         # always a discount
average_money(customer_transition, LOYAL, [FULL_PRICE, DISCOUNT, FULL_PRICE])     # a discount only when wavering

**Reading the result.** Always full price brings the most money in every single week and the least over the
quarter, \$52.51: the customer leaves. Always a discount keeps the customer, \$91.00. A discount only for a wavering
customer earns the most, \$121.53.

💭 **Think first.** At which question would you stop?

In [ ]:
average_money(quiz_transition, Q1, [ANSWER, STOP, STOP, STOP, STOP])           # take the first $100
average_money(quiz_transition, Q1, [ANSWER, ANSWER, STOP, STOP, STOP])         # take $200
average_money(quiz_transition, Q1, [ANSWER, ANSWER, ANSWER, STOP, STOP])       # take $400
average_money(quiz_transition, Q1, [ANSWER, ANSWER, ANSWER, ANSWER, ANSWER])   # answer all four

**Reading the result.** Taking the first \$100 home brings \$90.28 on average, and answering all four questions \$138.32.
The best of the four answers three questions and takes the \$400 home: \$173.12.

## What the three have in common

Over a quarter, and over a whole quiz, the action that brings the most money at once loses in each MDP: SELL the
goose, FULL_PRICE for every customer, STOP at the first \$100. An action brings money, and it also decides the
next state. The next lecture, runs and cumulative rewards, puts a number on a whole game.

Try a strategy of your own in the cell below, or another number of weeks for the goose: from how many weeks on
does keeping the goose beat the butcher?

In [ ]:
average_money(goose_transition, GOOSE, [KEEP, KEEP], turns=7)   # change me: the strategy, or the number of weeks